# 05 — Export to Excel

Compiles all data, features, and model findings into a single multi-sheet Excel workbook.
Run notebooks 01–04 first to ensure all processed files are up to date.

Output: `data/processed/ivy_football_analysis.xlsx`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.models import (
    build_game_dataset, train_roster_model,
    run_scheme_regression
)
import statsmodels.formula.api as smf

OUT_PATH = '../data/processed/ivy_football_analysis.xlsx'
print('Loading processed data...')

master    = pd.read_csv('../data/processed/master_labeled.csv')
schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
rosters   = pd.read_csv('../data/raw/rosters/rosters_raw.csv')
game_df   = build_game_dataset(master, schedules)

print(f'Master: {master.shape} | Game rows: {game_df.shape[0]} | Roster rows: {len(rosters)}')

In [ ]:
# ── Run all analyses so results are available for export ──────────────────────

# Scheme regression
reg_result = run_scheme_regression(master)
coef_df = reg_result['coef_df'].copy()
coef_df['significant'] = coef_df['pvalue'] < 0.05

# Win % by scheme (raw)
off_summary = (
    master.groupby('off_scheme')['ivy_win_pct']
    .agg(avg_win_pct='mean', std='std', n='count')
    .sort_values('avg_win_pct', ascending=False)
    .reset_index()
)
def_summary = (
    master.groupby('def_scheme')['ivy_win_pct']
    .agg(avg_win_pct='mean', std='std', n='count')
    .sort_values('avg_win_pct', ascending=False)
    .reset_index()
)

# Style correlations
style_cols = ['pass_tendency','tempo','spread_factor','explosiveness','front_heaviness']
available  = [c for c in style_cols if c in master.columns and master[c].notna().sum() > 5]
corrs = master[available + ['ivy_win_pct']].corr()['ivy_win_pct'].drop('ivy_win_pct')
n_per = {c: master[c].notna().sum() for c in available}
style_corr_df = pd.DataFrame({
    'feature': corrs.index,
    'pearson_r': corrs.values,
    'n': [n_per[c] for c in corrs.index],
    'note': [
        '2022-2024 only (box scores)' if c in ['pass_tendency','tempo','explosiveness']
        else '2014-2024 (roster data)'
        for c in corrs.index
    ]
}).sort_values('pearson_r', ascending=False).reset_index(drop=True)

# Random forest
rf_result = train_roster_model(game_df, model_type='rf')
rf_imp = rf_result['importances'].reset_index()
rf_imp.columns = ['feature', 'importance']

# Logistic regression model
lr_result = train_roster_model(game_df, model_type='lr')
lr_imp = lr_result['importances'].reset_index()
lr_imp.columns = ['feature', 'abs_coefficient']

# Model performance summary
model_perf = pd.DataFrame([
    {'model': 'Random Forest',      'cv_auc_mean': rf_result['cv_auc_mean'], 'cv_auc_std': rf_result['cv_auc_std'], 'n_features': len(rf_result['feature_cols'])},
    {'model': 'Logistic Regression','cv_auc_mean': lr_result['cv_auc_mean'], 'cv_auc_std': lr_result['cv_auc_std'], 'n_features': len(lr_result['feature_cols'])},
])

# Statsmodels logit results
logit_rows = []
for feature, formula in [
    ('home_OL_avg_weight',  'win ~ home_OL_avg_weight'),
    ('home_explosiveness',  'win ~ home_explosiveness'),
    ('home_DB_avg_weight',  'win ~ home_DB_avg_weight'),
    ('home_DL_avg_weight',  'win ~ home_DL_avg_weight'),
    ('home_pass_tendency',  'win ~ home_pass_tendency'),
]:
    sub = game_df[['win', feature]].dropna()
    if len(sub) < 30:
        continue
    try:
        m = smf.logit(formula, data=sub).fit(disp=0)
        p = m.params[feature]; se = m.bse[feature]; pv = m.pvalues[feature]
        logit_rows.append({'feature': feature, 'coef': round(p, 4),
                           'std_err': round(se, 4), 'pvalue': round(pv, 4),
                           'significant': pv < 0.05, 'n': len(sub)})
    except Exception:
        pass

# Combined logit
sub3 = game_df[['win','home_OL_avg_weight','home_explosiveness','home_DB_avg_weight']].dropna()
if len(sub3) >= 30:
    m3 = smf.logit('win ~ home_OL_avg_weight + home_explosiveness + home_DB_avg_weight', data=sub3).fit(disp=0)
    for feat in ['home_OL_avg_weight','home_explosiveness','home_DB_avg_weight']:
        logit_rows.append({'feature': f'{feat} (combined)', 'coef': round(m3.params[feat], 4),
                           'std_err': round(m3.bse[feat], 4), 'pvalue': round(m3.pvalues[feat], 4),
                           'significant': m3.pvalues[feat] < 0.05, 'n': len(sub3)})
logit_df = pd.DataFrame(logit_rows)

# Data coverage: which team-years have which data types
coverage = master[['school','year']].copy()
coverage['has_box_scores']    = master['totalYards'].notna()
coverage['has_roster']        = master['OL_avg_weight'].notna()
coverage['has_pass_tendency'] = master['pass_tendency'].notna()
coverage['has_spread_factor'] = master['spread_factor'].notna()
coverage['has_explosiveness'] = master['explosiveness'].notna()

print('All analyses complete — ready to export.')

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Helper functions ──────────────────────────────────────────────────────────

HEADER_FILL   = PatternFill('solid', fgColor='1F4E79')
SECTION_FILL  = PatternFill('solid', fgColor='BDD7EE')
SIG_FILL      = PatternFill('solid', fgColor='E2EFDA')   # green tint = significant
WARN_FILL     = PatternFill('solid', fgColor='FCE4D6')   # orange tint = watch out
HEADER_FONT   = Font(bold=True, color='FFFFFF', size=10)
SECTION_FONT  = Font(bold=True, color='1F4E79', size=10)
THIN          = Side(style='thin', color='BFBFBF')
BORDER        = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def style_header_row(ws, row, ncols):
    for col in range(1, ncols + 1):
        c = ws.cell(row=row, column=col)
        c.fill   = HEADER_FILL
        c.font   = HEADER_FONT
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = BORDER

def style_data_rows(ws, start_row, end_row, ncols):
    for r in range(start_row, end_row + 1):
        for col in range(1, ncols + 1):
            c = ws.cell(row=r, column=col)
            c.border    = BORDER
            c.alignment = Alignment(vertical='center')

def autofit(ws, min_w=8, max_w=40):
    for col in ws.columns:
        length = max(
            (len(str(cell.value)) if cell.value is not None else 0)
            for cell in col
        )
        ws.column_dimensions[get_column_letter(col[0].column)].width = \
            min(max(length + 2, min_w), max_w)

def write_df(ws, df, start_row=1, start_col=1, title=None):
    """Write a DataFrame to a worksheet with styled headers. Returns next free row."""
    r = start_row
    if title:
        c = ws.cell(row=r, column=start_col, value=title)
        c.font   = SECTION_FONT
        c.fill   = SECTION_FILL
        ws.merge_cells(start_row=r, start_column=start_col,
                       end_row=r,   end_column=start_col + len(df.columns) - 1)
        r += 1
    # Header
    for j, col in enumerate(df.columns, start_col):
        ws.cell(row=r, column=j, value=col)
    style_header_row(ws, r, len(df.columns))
    r += 1
    # Data
    for _, row_data in df.iterrows():
        for j, val in enumerate(row_data, start_col):
            cell = ws.cell(row=r, column=j)
            cell.value = None if (isinstance(val, float) and np.isnan(val)) else val
            if isinstance(val, float) and not np.isnan(val):
                cell.number_format = '0.000'
        r += 1
    style_data_rows(ws, start_row + (2 if title else 1), r - 1, len(df.columns))
    return r

print('Helper functions defined.')

In [ ]:
# ── Build the workbook ────────────────────────────────────────────────────────

# Select key columns for master sheet (not every intermediate column)
MASTER_COLS = [
    'school','year','off_scheme','def_scheme',
    'ivy_win_pct','win_pct','games','wins',
    'points_per_game','points_allowed_per_game','avg_margin',
    'OL_avg_weight','DL_avg_weight','LB_avg_weight','DB_avg_weight',
    'SKILL_avg_weight','OL_avg_height','SKILL_avg_height',
    'DL_to_LB_weight_ratio','OL_vs_DL_weight_diff','roster_depth',
    'pass_tendency','tempo','spread_factor','explosiveness','front_heaviness',
    'pass_rate','rush_rate','yards_per_play','turnovers_per_game',
    'totalYards','rushingYards','netPassingYards','passingTDs','rushingTDs',
]
MASTER_COLS = [c for c in MASTER_COLS if c in master.columns]

GAME_COLS = [
    'school','year','date','opponent','result','points','opp_points',
    'home_away','conference_game','win',
    'home_OL_avg_weight','home_DL_avg_weight','home_DB_avg_weight',
    'home_SKILL_avg_weight','home_roster_depth',
    'home_pass_tendency','home_tempo','home_spread_factor',
    'home_explosiveness','home_front_heaviness',
    'opp_run_heavy','opp_pass_heavy',
    'opp_OL_avg_weight','opp_DL_avg_weight','opp_pass_tendency',
    'opp_front_heaviness',
]
GAME_COLS = [c for c in GAME_COLS if c in game_df.columns]

with pd.ExcelWriter(OUT_PATH, engine='openpyxl') as writer:

    # ── Sheet 1: Overview ─────────────────────────────────────────────────────
    overview_rows = [
        ('Ivy Football Analytics — Overview', ''),
        ('', ''),
        ('DATA COVERAGE', ''),
        ('Years analysed', '2014–2024 (excl. 2020 — no season)'),
        ('Schools', 'Brown, Columbia, Cornell, Dartmouth, Harvard, Penn, Princeton, Yale'),
        ('Master dataset rows', len(master)),
        ('Game dataset rows', len(game_df)),
        ('Roster rows', len(rosters)),
        ('', ''),
        ('DATA NOTES', ''),
        ('Box score data', 'CFBD API — available 2022–2024 for all Ivy teams'),
        ('Roster data', 'Scraped from athletic sites — 2014–2024 (Cornell excluded: site returns current roster for all years)'),
        ('pass_tendency / tempo / explosiveness', 'Only populated 2022–2024 (requires box scores)'),
        ('spread_factor / front_heaviness', 'Populated 2014–2024 from roster weight ratios'),
        ('aggression / coverage_depth', 'Not available — CFBD does not provide per-play defensive stats for FCS teams'),
        ('Scheme labels 2014–2021', 'Based on spread_factor + front_heaviness only (roster data). Treat cautiously.'),
        ('Scheme labels 2022–2024', 'Based on full set: pass_tendency, tempo, spread_factor, explosiveness, front_heaviness'),
        ('', ''),
        ('MODEL PERFORMANCE', ''),
        ('Random Forest CV AUC', f"{rf_result['cv_auc_mean']:.3f} ± {rf_result['cv_auc_std']:.3f}"),
        ('Logistic Regression CV AUC', f"{lr_result['cv_auc_mean']:.3f} ± {lr_result['cv_auc_std']:.3f}"),
        ('Training rows (after dropna)', game_df[rf_result['feature_cols'] + ['win']].dropna().shape[0]),
        ('', ''),
        ('KEY FINDING', ''),
        ('Strongest predictor', 'home_explosiveness (yards/play) — p=0.001 in logit, top feature in both RF and LR'),
        ('Opponent type signal', 'opp_run_heavy is the #2 LR feature after the case-matching fix (2022–2024 Ivy games only)'),
        ('Scheme effects', 'No significant scheme effects found — likely reflects label inconsistency pre-2022 as much as true null effect'),
        ('', ''),
        ('SHEETS IN THIS WORKBOOK', ''),
        ('1. Overview', 'This sheet'),
        ('2. Master Dataset', 'One row per team-season (key columns)'),
        ('3. Game Dataset', 'One row per game from each Ivy team perspective'),
        ('4. Roster Summary', 'Physical feature averages by team-year'),
        ('5. Data Coverage', 'Which team-years have which data types'),
        ('6. Scheme Win Rates', 'Raw Ivy win% by offensive and defensive scheme'),
        ('7. Panel Regression', 'OLS regression: ivy_win_pct ~ scheme + school FE + year FE'),
        ('8. Style Correlations', 'Pearson r of each style score with Ivy win%'),
        ('9. RF Importances', 'Random Forest feature importances (top 20)'),
        ('10. LR Coefficients', 'Logistic Regression |coefficient| ranking (top 20)'),
        ('11. Logit Results', 'Statsmodels single-feature and combined logit tables'),
    ]
    overview_df = pd.DataFrame(overview_rows, columns=['Item', 'Value'])
    overview_df.to_excel(writer, sheet_name='Overview', index=False)

    # ── Sheet 2: Master Dataset ───────────────────────────────────────────────
    master[MASTER_COLS].to_excel(writer, sheet_name='Master Dataset', index=False)

    # ── Sheet 3: Game Dataset ─────────────────────────────────────────────────
    game_df[GAME_COLS].to_excel(writer, sheet_name='Game Dataset', index=False)

    # ── Sheet 4: Roster Summary ───────────────────────────────────────────────
    roster_cols = [
        'school','year',
        'OL_avg_weight','OL_avg_height','OL_n',
        'DL_avg_weight','DL_avg_height','DL_n',
        'LB_avg_weight','LB_avg_height','LB_n',
        'DB_avg_weight','DB_avg_height','DB_n',
        'SKILL_avg_weight','SKILL_avg_height','SKILL_n',
        'DL_to_LB_weight_ratio','OL_vs_DL_weight_diff',
        'roster_depth','upperclassman_ratio',
    ]
    roster_cols = [c for c in roster_cols if c in master.columns]
    master[roster_cols].to_excel(writer, sheet_name='Roster Summary', index=False)

    # ── Sheet 5: Data Coverage ────────────────────────────────────────────────
    coverage.to_excel(writer, sheet_name='Data Coverage', index=False)

    # ── Sheet 6: Scheme Win Rates ─────────────────────────────────────────────
    scheme_book = pd.concat([
        off_summary.assign(type='Offensive scheme').rename(columns={'off_scheme':'scheme'}),
        def_summary.assign(type='Defensive scheme').rename(columns={'def_scheme':'scheme'}),
    ], ignore_index=True)[['type','scheme','avg_win_pct','std','n']]
    scheme_book.to_excel(writer, sheet_name='Scheme Win Rates', index=False)

    # ── Sheet 7: Panel Regression ─────────────────────────────────────────────
    reg_export = coef_df[['term','coef','ci_lo','ci_hi','pvalue','significant']].copy()
    reg_export.columns = ['Term','Coefficient','CI Low (95%)','CI High (95%)','P-value','Significant (p<0.05)']
    reg_export.to_excel(writer, sheet_name='Panel Regression', index=False)

    # ── Sheet 8: Style Correlations ───────────────────────────────────────────
    style_corr_df.to_excel(writer, sheet_name='Style Correlations', index=False)

    # ── Sheet 9: RF Importances ───────────────────────────────────────────────
    rf_imp.head(20).to_excel(writer, sheet_name='RF Importances', index=False)

    # ── Sheet 10: LR Coefficients ─────────────────────────────────────────────
    lr_imp.head(20).to_excel(writer, sheet_name='LR Coefficients', index=False)

    # ── Sheet 11: Logit Results ───────────────────────────────────────────────
    logit_df.to_excel(writer, sheet_name='Logit Results', index=False)

print('Written. Now applying formatting...')

# ── Post-write formatting ─────────────────────────────────────────────────────
wb = load_workbook(OUT_PATH)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Style header row
    max_col = ws.max_column
    style_header_row(ws, 1, max_col)
    ws.freeze_panes = 'A2'

    # Highlight significant rows in regression and logit sheets
    if sheet_name in ('Panel Regression', 'Logit Results'):
        sig_col = None
        for j in range(1, max_col + 1):
            if ws.cell(1, j).value in ('Significant (p<0.05)', 'significant'):
                sig_col = j
        if sig_col:
            for r in range(2, ws.max_row + 1):
                if ws.cell(r, sig_col).value is True:
                    for col in range(1, max_col + 1):
                        ws.cell(r, col).fill = SIG_FILL

    # Bold the overview section headers
    if sheet_name == 'Overview':
        section_keywords = {
            'DATA COVERAGE','DATA NOTES','MODEL PERFORMANCE',
            'KEY FINDING','SHEETS IN THIS WORKBOOK',
            'Ivy Football Analytics — Overview'
        }
        for r in range(2, ws.max_row + 1):
            val = ws.cell(r, 1).value
            if val in section_keywords:
                for col in range(1, 3):
                    ws.cell(r, col).font = SECTION_FONT
                    ws.cell(r, col).fill = SECTION_FILL

    autofit(ws)

# Freeze first two columns on large data sheets
for sn in ('Master Dataset', 'Game Dataset', 'Roster Summary', 'Data Coverage'):
    wb[sn].freeze_panes = 'C2'

wb.save(OUT_PATH)
print(f'Saved: {OUT_PATH}')
print(f'Sheets: {wb.sheetnames}')